# Olist Delivery Delays & Review Scores: Data Cleaning & Preparation

**Business Objective:** Evaluate how delivery delays affect customer satisfaction, and identify actionable ways for Olist to reduce late deliveries and protect its review scores.

**Overview:** The raw Olist dataset consists of multiple separate CSV files containing data quality issues such as duplicate reviews, missing category translations, and inconsistent timestamps. This notebook cleans and standardizes the raw data, engineers delivery performance features, and exports structured tables to MySQL for downstream SQL analysis and Power BI dashboarding.

**Workflow Summary**
1. Inspect the raw datasets for missing values, data types, and exact duplicates.
2. Resolve data quality issues: deduplicate reviews, add missing category translations, and handle contradictory delivery records and negative durations.
3. Flag invalid orders (cancelled, outside the target date range, or missing item records) rather than deleting them to preserve data lineage.
4. Engineer delivery metrics: `is_late`, `delivery_days`, `delivery_delay_days`, `seller_handling_days`, and `transit_days`.
5. Export the cleaned tables to MySQL (`olist_db`).

**Key Data Definitions & Rules**
- `is_valid_order = 1` represents a completed, valid purchase: not cancelled, ordered between Jan 2017 and Aug 2018, and containing at least one item.
- `in_delay_analysis = 1` identifies valid orders that were successfully delivered and have verified delay metrics. This serves as the primary filter for all delay and review-score analyses.
- Delivery metrics are left as `NULL`/`NA` (rather than zero) when an order was never delivered or contains contradictory timestamps, preventing artificial distortion of delivery averages.
- No rows are deleted from `orders`; problematic records are isolated using binary flags for full auditability.

**Environment & Prerequisites:** Requires `pandas`, `numpy`, `sqlalchemy`, and `pymysql`, along with a local MySQL database named `olist_db` and the raw Olist CSV files in the working directory.

## 1. Importing libraries

`pandas` is used for tabular data manipulation and `numpy` for vectorized calculations. `SQLAlchemy` is imported in Step 19 for the MySQL export.

In [1]:
import pandas as pd
import numpy as np

## 2. Loading source datasets

The raw CSV files are loaded into separate pandas DataFrames: `orders`, `order_items`, `payments`, `reviews`, `customers`, `products`, `sellers`, and `translation`.

**Schema Relationships:**
- `orders` joins to `customers` on `customer_id`.
- `orders` joins to `order_items`, `payments`, and `reviews` on `order_id`.
- `order_items` joins to `products` on `product_id` and to `sellers` on `seller_id`.

In [2]:
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

## 3. Checking order data quality

An initial inspection of the `orders` table establishes a baseline for column data types, missing values, and duplicate rows.

Missing values in the delivery timestamp columns are expected: approximately 3,000 orders have no customer delivery date because they were never delivered (e.g., cancelled or still in transit). The table contains no duplicate rows.

In [3]:
orders.info()
print("--- ORDERS ---")
print("Missing Values:\n", orders.isnull().sum())
print("Duplicates:", orders.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB
--- ORDERS ---
Missing Values:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_d

## 4. Checking order-item data quality

The `order_items` table contains one row per item purchased within an order and links orders to specific products and sellers. Quality checks confirm there are no missing values or duplicate records.

In [4]:
order_items.info()
print("\n--- ITEMS ---")
print("Missing Values:\n", order_items.isnull().sum())
print("Duplicates:", order_items.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 18.4 MB

--- ITEMS ---
Missing Values:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Duplicates: 0


## 5. Checking payment data quality

The `payments` table has no missing values or duplicate rows. Three records have a payment type of `not_defined`; these are retained in the cleaned table and can be filtered out in downstream reporting if needed.

In [5]:
payments.info()
print("\n--- PAYMENTS ---")
print("Missing Values:\n", payments.isnull().sum())
print("Duplicates:", payments.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB

--- PAYMENTS ---
Missing Values:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Duplicates: 0


## 6. Checking review data quality

In the `reviews` table, comment titles and messages are optional fields and contain a high volume of missing values. Because text fields are outside the scope of the quantitative delay analysis, both columns are dropped in Step 13.

While there are no exact duplicate rows, several orders have multiple review records attached to them. This one-to-many issue is resolved in Step 13.

In [6]:
reviews.info()
print("\n--- REVIEWS ---")
print("Missing Values:\n", reviews.isnull().sum())
print("Duplicates:", reviews.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB

--- REVIEWS ---
Missing Values:
 review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
Duplicates: 0


## 7. Checking customer data quality

The `customers` table contains no missing values or duplicate rows.

Because Olist generates a new `customer_id` for every transaction, this table contains one row per order rather than one row per unique person (repeat buyers are tracked via `customer_unique_id`).

In [7]:
customers.info()
print("\n--- CUSTOMERS ---")
print("Missing Values:\n", customers.isnull().sum())
print("Duplicates:", customers.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

--- CUSTOMERS ---
Missing Values:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplicates: 0


## 8. Checking product data quality

Inspection of the `products` table highlights two data quality issues: 610 products are missing a category name (as well as name length, description length, and photo count), and 2 products are missing weight and dimension measurements. Missing categories are handled in Step 14.

In [8]:
products.info()
print("\n--- PRODUCTS ---")
print("Missing Values:\n", products.isnull().sum())
print("Duplicates:", products.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB

--- PRODUCTS ---
Missing Values:
 product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g       

## 9. Checking seller data quality

The `sellers` table (3,095 records) is verified for completeness and contains no missing values or duplicate rows.

In [9]:
sellers.info()
print("\n--- SELLERS ---")
print("Missing Values:\n", sellers.isnull().sum())
print("Duplicates:", sellers.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB

--- SELLERS ---
Missing Values:
 seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64
Duplicates: 0


## 10. Checking category translation data quality

The category translation table maps Portuguese product category names to English. While the table has no nulls or duplicates, two categories present in the `products` dataset are missing from this mapping and are added manually in Step 14.

In [10]:
translation.info()
print("\n--- TRANSLATION ---")
print("Missing Values:\n", translation.isnull().sum())
print("Duplicates:", translation.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB

--- TRANSLATION ---
Missing Values:
 product_category_name            0
product_category_name_english    0
dtype: int64
Duplicates: 0


## 11. Converting order timestamps

Timestamp columns are imported from CSV as plain strings. Here, purchase, approval, carrier pickup, customer delivery, and estimated delivery columns are converted to `datetime` objects to support date range filtering and duration calculations.

In [11]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

## 12. Flagging orders excluded from analysis

Rather than deleting problematic records, binary flag columns are created so excluded orders remain auditable:
- `is_canceled`: order status is `canceled` (625 orders).
- `in_date_range`: purchase occurred between Jan 2017 and Aug 2018, excluding sparse boundary months (349 orders fall outside).
- `missing_items`: order has no matching rows in `order_items` (775 orders).
- `is_valid_order`: composite filter identifying orders that are not cancelled, fall within the target date range, and contain item records (97,905 valid orders).

In [12]:
orders['is_canceled'] = (orders['order_status'] == 'canceled').astype(int)
orders['in_date_range'] = ((orders['order_purchase_timestamp'] >= '2017-01-01') & (orders['order_purchase_timestamp'] < '2018-09-01')).astype(int)
orders['missing_items'] = (~orders['order_id'].isin(order_items['order_id'])).astype(int)
orders['is_valid_order'] = ((orders['is_canceled'] == 0) & (orders['in_date_range'] == 1) &(orders['missing_items'] == 0)).astype(int)

## 13. Deduplicating and simplifying reviews

Because some orders have multiple reviews (551 extra rows), the table is sorted by `review_answer_timestamp` and deduplicated to retain only the **latest** review per order.

Unneeded text and timestamp columns (`review_comment_title`, `review_comment_message`, `review_creation_date`, and `review_answer_timestamp`) are then dropped after sorting.

**Result:** 98,673 unique order reviews with columns `review_id`, `order_id`, and `review_score`.

In [13]:
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])
reviews.sort_values(['order_id', 'review_answer_timestamp'], inplace=True)
reviews.drop_duplicates(subset='order_id', keep='last', inplace=True)
reviews.drop(columns=['review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp'], inplace=True)
print(len(reviews)); print(reviews.columns.tolist())

98673
['review_id', 'order_id', 'review_score']


## 14. Translating and simplifying products

English category names are merged into the `products` table. Two categories missing from the translation table (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) are appended manually, and products without a category are assigned `unknown`. The merged column is renamed to `product_category`, and unused product dimension columns are dropped.

**Result:** 32,951 products (including 610 labeled `unknown`) with columns `product_id` and `product_category`.

In [14]:
missing_cats = pd.DataFrame({'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],'product_category_name_english': ['pc_gamer', 'portable_kitchen_food_preparers']})
translation_fixed = pd.concat([translation, missing_cats], ignore_index=True)
products = products.merge(translation_fixed, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)
products.drop(columns=['product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'], inplace=True)
print(len(products), (products['product_category'] == 'unknown').sum())   # 32951 610
print(products.columns.tolist())

32951 610
['product_id', 'product_category']


## 15. Creating time and delivery features

Core temporal and delivery metrics are engineered from the exact timestamps:
- `order_year` and `order_month`: extracted from the purchase timestamp for seasonality analysis.
- `delivery_days`: total completed days from purchase to customer delivery (rounded down).
- `delivery_delay_days`: days delivered past the estimated date (rounded up; zero or negative values indicate on-time or early delivery).
- `is_late`: binary indicator set to `1` if delivered after the estimated date, `0` if on time, and `NA` if never delivered.

`is_late` is computed from exact timestamp differences prior to integer rounding so fractional delays are accurately captured. `SECONDS_PER_DAY` is defined here and reused in Step 16.

In [15]:
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.month

SECONDS_PER_DAY = 60 * 60 * 24
delay_exact = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.total_seconds() / SECONDS_PER_DAY
days_exact = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / SECONDS_PER_DAY

orders['is_late'] = np.where(delay_exact > 0, 1, 0)
orders.loc[orders['order_delivered_customer_date'].isnull(), 'is_late'] = np.nan
orders['is_late'] = orders['is_late'].astype('Int64')

orders['delivery_days'] = np.floor(days_exact).astype('Int64')
orders['delivery_delay_days'] = np.ceil(delay_exact).astype('Int64')

## 16. Splitting delivery time into seller handling and carrier transit

Total delivery time is decomposed into two operational stages to isolate the root cause of delays:
- `seller_handling_days`: days from purchase until the seller hands the parcel to the carrier (identifies **seller-side** bottlenecks).
- `transit_days`: days from carrier pickup until final customer delivery (identifies **carrier-side** logistics bottlenecks).

Both metrics are rounded down to whole days. Separating these stages helps determine whether late deliveries stem primarily from slow seller fulfillment or slow carrier transit. Negative durations caused by timestamp errors are cleaned in Step 17.

In [16]:
orders['seller_handling_days'] = np.floor((orders['order_delivered_carrier_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / SECONDS_PER_DAY).astype('Int64')
orders['transit_days'] = np.floor((orders['order_delivered_customer_date'] - orders['order_delivered_carrier_date']).dt.total_seconds() / SECONDS_PER_DAY).astype('Int64')

## 17. Flagging inconsistent delivery records and impossible durations

Two categories of timestamp anomalies are resolved in this step:

**1. Contradictory delivery records (`inconsistent_delivery`):**
- Order status is `delivered`, but the delivery timestamp is missing (8 orders).
- A delivery timestamp is present, but the order status is not `delivered` (6 orders, all cancelled).

These 14 records are flagged with `inconsistent_delivery = 1`, and their delivery metrics (`is_late`, `delivery_days`, `delivery_delay_days`, `seller_handling_days`, and `transit_days`) are set to `NA` to prevent distortion of summary statistics.

**2. Impossible negative durations:**
- 166 orders have a negative `seller_handling_days` (carrier pickup timestamp precedes purchase timestamp).
- 23 orders have a negative `transit_days` (customer delivery timestamp precedes carrier pickup timestamp).

Only the invalid negative duration values are set to `NA`, preserving the remaining valid fields of each order.

**Result:** 7,826 late orders, 2,971 `NA` values in `is_late`, and zero negative durations remaining in `seller_handling_days` or `transit_days`.

In [17]:
delivered_no_date = (orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].isnull()
date_not_delivered = (orders['order_status'] != 'delivered') & orders['order_delivered_customer_date'].notna()
orders['inconsistent_delivery'] = (delivered_no_date | date_not_delivered).astype(int)
bad = orders['inconsistent_delivery'] == 1
orders.loc[bad, ['is_late', 'delivery_days', 'delivery_delay_days', 'seller_handling_days', 'transit_days']] = pd.NA

# negative durations are impossible, so treat them as bad data
neg_handling = (orders['seller_handling_days'] < 0).fillna(False)
neg_transit = (orders['transit_days'] < 0).fillna(False)
print("Negative handling:", neg_handling.sum(), "| Negative transit:", neg_transit.sum())
orders.loc[neg_handling, 'seller_handling_days'] = pd.NA
orders.loc[neg_transit, 'transit_days'] = pd.NA
print(orders[['seller_handling_days', 'transit_days']].min())

Negative handling: 166 | Negative transit: 23
seller_handling_days    0
transit_days            0
dtype: Int64


In [18]:
# quick check: averages of handling + transit should be close to delivery_days
print(orders[['delivery_days', 'seller_handling_days', 'transit_days']].describe())

       delivery_days  seller_handling_days  transit_days
count        96470.0               97479.0       96446.0
mean       12.093604              2.751034      8.881073
std          9.55138              3.581461      8.744552
min              0.0                   0.0           0.0
25%              6.0                   1.0           4.0
50%             10.0                   2.0           7.0
75%             15.0                   4.0          12.0
max            209.0                 125.0         205.0


## 18. Creating the delay-analysis flag

Because `is_valid_order = 1` still includes undelivered orders (such as those in `shipped` or `processing` status) that have no `is_late` value, a dedicated flag is created to streamline downstream SQL queries:

- `in_delay_analysis = 1`: identifies valid orders that were delivered and have a verified `is_late` value.

This flag is created after Step 17 so that inconsistent delivery records are automatically excluded.

**Result:** 96,203 analyzable orders, comprising 88,381 on-time deliveries and 7,822 late deliveries—establishing an overall **8.13% late-delivery rate**.

In [19]:
orders['in_delay_analysis'] = ((orders['is_valid_order'] == 1) & orders['is_late'].notna()).astype(int)
print(orders['in_delay_analysis'].sum())
print(orders[orders['in_delay_analysis'] == 1]['is_late'].value_counts())
print(orders[orders['in_delay_analysis'] == 1]['is_late'].mean())

96203
is_late
0    88381
1     7822
Name: count, dtype: Int64
0.08130723574108915


## 19. Saving clean tables to MySQL

The cleaned DataFrames are exported to the `olist_db` MySQL database using `SQLAlchemy` and `PyMySQL`. The database password is entered interactively via `getpass` so credentials are never hardcoded in the notebook.

In [20]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

password = getpass("MySQL password: ")

engine = create_engine(URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="olist_db"
))

orders.to_sql('orders', con=engine, if_exists='replace', index=False)
order_items.to_sql('order_items', con=engine, if_exists='replace', index=False)
payments.to_sql('payments', con=engine, if_exists='replace', index=False)
reviews.to_sql('reviews', con=engine, if_exists='replace', index=False)
customers.to_sql('customers', con=engine, if_exists='replace', index=False)
products.to_sql('products', con=engine, if_exists='replace', index=False)
sellers.to_sql('sellers', con=engine, if_exists='replace', index=False)
print("saved")

saved
